# 04 — Generated Route Evaluation

## Why evaluate generated routes?

In language modeling, we evaluate generated text using metrics like BLEU, ROUGE, or perplexity. For climbing routes, we need domain-specific evaluation:

1. **Validity**: Does the route follow the rules of climbing boards?
2. **Novelty**: Is the route different from existing climbs, or just a copy?
3. **Geometric plausibility**: Are the holds in reasonable positions?
4. **Grade consistency**: Does the route's predicted grade match the requested grade?

### Validity checks

A "basic valid" route must have:
- At least 3 holds (you need at least 2 hands + 1 foot to climb)
- No duplicate placements (you can't use the same hold twice)
- At least one start hold and one finish hold
- All holds from the same board (no mixing TB2 and Kilter holds)

A "strict valid" route additionally has:
- At least one middle hold (most real climbs have more than just start + finish)
- At least 4 holds total

### Novelty metrics

We measure novelty using **Jaccard distance**: 1 minus the Jaccard similarity between the generated route's hold set and the most similar real route's hold set.

- Jaccard similarity = |A intersection B| / |A union B|
- Novelty distance = 1 - Jaccard similarity

A novelty distance of 1.0 means the generated route shares no holds with any real route. A distance of 0.0 means it's identical to an existing route.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from climbingboardgpt.evaluation import (
    build_placement_coords,
    frames_to_holds,
    holds_to_placement_set,
    nearest_real_route_same_board,
    parse_token_list,
    simple_route_features,
    tokens_to_hold_records,
    validity_from_records,
)
from climbingboardgpt.grades import to_grouped_v
from climbingboardgpt.models import JointRouteTransformerRegressor

In [ ]:
# Load generated routes and real routes for comparison
# NOTE: This notebook requires that you've run notebook 03 first to
# generate and save the routes.

TOKENIZED = ROOT / "data" / "processed" / "tokenized"
GENERATED = ROOT / "data" / "processed" / "generation"

# Check if required files exist
generated_path = GENERATED / "generated_routes.csv"
routes_path = TOKENIZED / "route_sequences.csv"
token_meta_path = TOKENIZED / "token_metadata.csv"

if not generated_path.exists():
    raise FileNotFoundError(
        f"Generated routes not found at: {generated_path}\n"
        f"Please run notebook 03 first to generate and save routes,\n"
        f"or run: python scripts/03_train_route_generator.py"
    )

if not routes_path.exists() or not token_meta_path.exists():
    raise FileNotFoundError(
        f"Tokenized data not found at: {TOKENIZED}\n"
        f"Please run notebook 01 first to tokenize routes,\n"
        f"or run: python scripts/01_tokenize_routes.py"
    )

df_generated = pd.read_csv(generated_path)
df_real = pd.read_csv(routes_path)
df_token_meta = pd.read_csv(token_meta_path)

print(f"Generated routes: {len(df_generated):,}")
print(f"Real routes: {len(df_real):,}")

## Parse generated tokens and check validity

We parse the generated token sequences and check each route for validity.

In [ ]:
# Parse the token strings into structured records
df_generated["tokens_parsed"] = df_generated["tokens"].apply(parse_token_list)

# Extract hold information from tokens
df_generated["hold_records"] = df_generated["tokens_parsed"].apply(tokens_to_hold_records)

# Check validity for each generated route
validity = pd.DataFrame(df_generated["hold_records"].apply(validity_from_records).tolist())
df_eval = pd.concat([df_generated.reset_index(drop=True), validity], axis=1)

print("Validity rates by board:")
print("=" * 50)
validity_summary = df_eval.groupby("board_key").agg(
    total=("basic_valid_eval", "count"),
    basic_valid_rate=("basic_valid_eval", "mean"),
    strict_valid_rate=("strict_valid_eval", "mean"),
    avg_holds=("n_holds_eval", "mean"),
).round(3)
print(validity_summary)

## Novelty against real climbs

For each generated route, we find the most similar real route from the same board (by Jaccard similarity of hold sets). A good generator should produce routes that are novel (low Jaccard similarity to existing routes) while still being valid.

In [ ]:
# Convert hold sets to frozensets for fast comparison
df_eval["hold_set"] = df_eval["hold_records"].apply(
    lambda records: frozenset(int(record["placement_id"]) for record in records)
)

# Parse real routes' frames strings into hold sets
df_real["real_holds"] = df_real["frames"].apply(frames_to_holds)
df_real["hold_set"] = df_real["real_holds"].apply(holds_to_placement_set)

# Find nearest real route for each generated route
print("Computing novelty (finding nearest real route for each generated route)...")
print("This may take a few minutes...")

nearest = pd.DataFrame(
    df_eval.apply(
        lambda row: nearest_real_route_same_board(
            generated_set=row["hold_set"],
            generated_board_key=row["board_key"],
            real_df=df_real,
        ),
        axis=1,
    ).tolist()
)
df_eval = pd.concat([df_eval, nearest], axis=1)

print("\nNovelty statistics by board:")
print("=" * 50)
novelty_summary = df_eval.groupby("board_key").agg(
    mean_jaccard=("nearest_real_jaccard", "mean"),
    mean_novelty=("novelty_distance", "mean"),
    median_novelty=("novelty_distance", "median"),
).round(3)
print(novelty_summary)

## Geometric descriptors

We compute simple geometric features for each generated route:

- `geom_n_holds`: Number of holds
- `geom_height`: Vertical extent of the route
- `geom_width`: Horizontal extent
- `geom_mean_hand_reach`: Average distance between hand holds

These features help us understand whether generated routes have reasonable spatial properties.

In [ ]:
# Build coordinate lookup from token metadata
coords = build_placement_coords(df_token_meta)

# Compute geometric features for each generated route
geom = pd.DataFrame(
    df_eval.apply(
        lambda row: simple_route_features(
            board_key=row["board_key"],
            records=row["hold_records"],
            placement_coords=coords,
        ),
        axis=1,
    ).tolist()
)
df_eval = pd.concat([df_eval, geom], axis=1)

print("Geometric feature statistics by board:")
print("=" * 50)
geom_summary = df_eval.groupby("board_key").agg(
    mean_holds=("geom_n_holds", "mean"),
    mean_height=("geom_height", "mean"),
    mean_width=("geom_width", "mean"),
    mean_hand_reach=("geom_mean_hand_reach", "mean"),
).round(3)
print(geom_summary)

## Grade consistency (using the trained critic)

If we have a trained grade predictor (from notebook 02), we can use it as a **critic** to check whether generated routes have grades consistent with what was requested.

This is similar to how GANs use a discriminator to evaluate generated samples, except our critic is a regression model rather than a binary classifier.

In [ ]:
# Try to load the grade critic from notebook 02
GRADE_MODEL_PATH = ROOT / "models" / "joint_transformer_grade_predictor.pth"

def load_grade_critic(model_path, device):
    """Load the trained grade predictor model."""
    if not model_path.exists():
        return None
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)

    cfg = checkpoint["config"]
    stoi = {str(k): int(v) for k, v in checkpoint["stoi"].items()}
    coord_features = checkpoint["coord_features"]
    if not isinstance(coord_features, torch.Tensor):
        coord_features = torch.tensor(coord_features, dtype=torch.float32)

    model = JointRouteTransformerRegressor(
        vocab_size=cfg["vocab_size"],
        max_len=cfg["max_len"],
        coord_features=coord_features,
        d_model=cfg.get("d_model", 128),
        nhead=cfg.get("nhead", 4),
        num_layers=cfg.get("num_layers", 4),
        dim_feedforward=cfg.get("dim_feedforward", 256),
        dropout=cfg.get("dropout", 0.10),
        pad_id=cfg.get("pad_id", stoi["<PAD>"]),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return {
        "model": model,
        "stoi": stoi,
        "pad_id": stoi["<PAD>"],
        "unk_id": stoi["<UNK>"],
        "max_len": cfg["max_len"],
    }


def predict_generated_grade(tokens, critic, device):
    """Predict the difficulty of a generated route using the critic."""
    model = critic["model"]
    stoi = critic["stoi"]
    pad_id = critic["pad_id"]
    unk_id = critic["unk_id"]
    max_len = critic["max_len"]

    # Remove grade tokens and replace BOS with CLS
    tokens = [t for t in tokens if not t.startswith("<GRADE_")]
    if tokens and tokens[0] == "<BOS>":
        tokens = ["<CLS>"] + tokens[1:]
    else:
        tokens = ["<CLS>"] + tokens

    ids = [stoi.get(t, unk_id) for t in tokens][:max_len]
    mask = [1] * len(ids)
    if len(ids) < max_len:
        pad_n = max_len - len(ids)
        ids += [pad_id] * pad_n
        mask += [0] * pad_n

    with torch.no_grad():
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
        attention_mask = torch.tensor([mask], dtype=torch.bool, device=device)
        return float(model(input_ids, attention_mask).cpu().item())


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
critic = load_grade_critic(GRADE_MODEL_PATH, device)

if critic is not None:
    print("Grade critic loaded successfully!")
    print(f"Device: {device}")
else:
    print("No trained grade critic found. Skipping critic-based scoring.")
    print("Run notebook 02 first to train the grade predictor.")

In [ ]:
# Apply the critic to evaluate grade consistency
if critic is not None:
    df_eval["critic_pred_display_difficulty"] = df_eval["tokens_parsed"].apply(
        lambda tokens: predict_generated_grade(tokens, critic, device)
    )
    df_eval["critic_pred_grouped_v"] = df_eval["critic_pred_display_difficulty"].apply(to_grouped_v)
    df_eval["critic_v_error"] = df_eval["critic_pred_grouped_v"] - df_eval["requested_grouped_v"]

    print("Grade consistency by board:")
    print("=" * 50)
    critic_summary = df_eval.groupby("board_key").agg(
        exact_v=("critic_v_error", lambda s: float((s == 0).mean() * 100)),
        within_1_v=("critic_v_error", lambda s: float((s.abs() <= 1).mean() * 100)),
        within_2_v=("critic_v_error", lambda s: float((s.abs() <= 2).mean() * 100)),
        mean_error=("critic_v_error", "mean"),
    ).round(2)
    print(critic_summary)
else:
    print("Skipping critic evaluation (no model loaded).")

## Ranking generated routes

We rank candidates by a composite score that rewards:
- **Basic validity** (required): At least 3 holds, start/finish, no duplicates, one board
- **Strict validity** (bonus): Also has middle holds and 4+ holds
- **Novelty** (higher is better): Distance from nearest real route
- **Grade consistency** (if critic available): Predicted grade close to requested grade

In [ ]:
# Rank candidates by composite score
ranked = df_eval.copy()
ranked["score"] = 0.0
ranked["score"] += ranked["basic_valid_eval"].astype(float) * 2.0
ranked["score"] += ranked["strict_valid_eval"].astype(float) * 1.0
ranked["score"] += ranked["novelty_distance"].fillna(0.0)

if "critic_v_error" in ranked.columns:
    ranked["score"] += (ranked["critic_v_error"].abs() <= 1).astype(float)
    ranked["score"] -= 0.25 * ranked["critic_v_error"].abs()

print("Top 10 generated routes by composite score:")
print("=" * 70)
top_routes = ranked.sort_values("score", ascending=False).head(10)
display_cols = ["board_key", "score", "basic_valid_eval", "strict_valid_eval", "novelty_distance"]
if "critic_v_error" in top_routes.columns:
    display_cols.append("critic_v_error")
print(top_routes[display_cols].to_string(index=False))

## Save evaluation results

We save the full evaluation DataFrame and the top candidates for further analysis.

In [ ]:
# Save evaluation results
OUT_DIR = ROOT / "data" / "processed" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_eval.to_csv(OUT_DIR / "generated_route_evaluation.csv", index=False)
top_candidates = ranked.sort_values("score", ascending=False).head(100)
top_candidates.to_csv(OUT_DIR / "top_generated_candidates.csv", index=False)

print(f"Saved evaluation results to: {OUT_DIR}")
print(f"  - generated_route_evaluation.csv ({len(df_eval)} rows)")
print(f"  - top_generated_candidates.csv (100 rows)")

print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"\nTotal generated routes: {len(df_eval):,}")
print(f"\nBasic validity rate: {df_eval['basic_valid_eval'].mean():.1%}")
print(f"Strict validity rate: {df_eval['strict_valid_eval'].mean():.1%}")
print(f"Mean novelty distance: {df_eval['novelty_distance'].mean():.3f}")

if 'critic_v_error' in df_eval.columns:
    print(f"\nGrade consistency:")
    print(f"  Exact V-grade: {(df_eval['critic_v_error'] == 0).mean():.1%}")
    print(f"  Within 1 V-grade: {(df_eval['critic_v_error'].abs() <= 1).mean():.1%}")
    print(f"  Within 2 V-grades: {(df_eval['critic_v_error'].abs() <= 2).mean():.1%}")
else:
    print("\n(Grade consistency not available - no critic model loaded)")

## Key Takeaways

1. **Validity**: The generator produces routes that mostly satisfy structural constraints (start/finish holds, no duplicates, single board).

2. **Novelty**: Generated routes are meaningfully different from existing routes, as measured by Jaccard distance.

3. **Geometric plausibility**: The geometric features (height, width, hand reach) should be in reasonable ranges compared to real routes.

4. **Grade consistency**: If the critic is available, we can check whether routes generated at a requested grade actually feel like that grade.

### Limitations

- Validity checks are structural, not semantic. A route might have valid start/finish holds but still be impossible.
- Geometric features are simple. More sophisticated analysis could check reachability and move sequences.
- The critic model was trained on real data, so it may not generalize well to novel route structures.